# Optuna with Scikit‑Learn Pipelines

This notebook demonstrates end‑to‑end hyperparameter optimization using **Optuna** with a reusable scikit‑learn pipeline.

[Optuna on PyPi](https://pypi.org/project/optuna/)

### Where Optuna Fits 

- Dataset arrives, customer wants a model  
  - You perform best model and scaler workbook
  - Once model and scaler identified:
    - Run Optuna to identify best hyperparameters for your model  

In [1]:
#%pip install optuna
#%pip install optuna-dashboard

## 1. Imports

In [2]:
import pandas as pd

from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.base import clone

import optuna
import optuna_dashboard


## 2. Load Sample Data

(Replace with your own dataset if needed.)

In [3]:
TRANSFORM_URL = 'https://raw.githubusercontent.com/Oldmage112/TI4-Project/refs/heads/main/PowerBI/Derived%20CSVs/Factions_Transform.csv'
WINNERS_URL   = 'https://raw.githubusercontent.com/Oldmage112/TI4-Project/refs/heads/main/PowerBI/Derived%20CSVs/Factions_Winners.csv'

df_transform = pd.read_csv(TRANSFORM_URL, index_col=-1)
df_winners   = pd.read_csv(WINNERS_URL, index_col=0)

# Merge on Index
df = df_transform.merge(df_winners, on='Index', how='inner')

# All columns are categorical — force every column to object dtype
# so dtype-based selectors never mis-classify integer-valued columns as numeric
df = df.astype(str)

target_col = 'WinningFaction'
X = df.drop(columns=[target_col])
y = df[target_col]

print(f'Shape: {X.shape}  |  Target classes: {y.unique()}')


Shape: (17797, 49)  |  Target classes: <StringArray>
[            'XxchaKingdom',            'KeleresMentak',
            'YssarilTribes',             'ArgentFlight',
          'Vuil'raithCabal',      'MahactGeneSorcerers',
           'GhostsofCreuss',                  'Arborec',
             'SardakkN'orr',             'L1Z1XMindnet',
               'TitansofUl',          'NaaluCollective',
          'EmiratesofHacan',               'NekroVirus',
           'YinBrotherhood',          'MentakCoalition',
        'NaazRokhaAlliance',                    'Nomad',
     'UniversitiesofJolNar',               'ClanofSaar',
          'FederationofSol',            'EmbersofMuaat',
                 'Empyrean',                    'Winnu',
             'KeleresXxcha',           'BaronyofLetnev',
            'KeleresArgent',                  'Keleres',
    'DeepwroughtScholarate',              'LastBastion',
                'Firmament',         'RalNelConsortium',
         'CrimsonRebellion',       

## 3. Preprocessing

In [4]:
# All features are categorical (including integer-valued ones),
# so we encode every column with OrdinalEncoder.
ordinal_enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

preprocessor = make_column_transformer(
    (ordinal_enc, list(X.columns)),  # encode ALL columns
    remainder='drop',
)



## 4. Baseline Pipeline

In [5]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler',       StandardScaler()),
    ('classifier',   GradientBoostingClassifier(random_state=42)),
])


## 5. Optuna Objective Function

In [6]:

def objective(trial):
    params = {
        'classifier__n_estimators':  trial.suggest_int('n_estimators', 50, 500),
        'classifier__max_depth':     trial.suggest_int('max_depth', 2, 10),
        'classifier__learning_rate': trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True),
        'classifier__subsample':     trial.suggest_float('subsample', 0.5, 1.0),
        'classifier__max_features':  trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'classifier__min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
    }

    model_trial = clone(model)
    model_trial.set_params(**params)

    scores = cross_val_score(
        model_trial,
        X,
        y,
        cv=5,
        scoring='accuracy',
    )

    return scores.mean()


## 6. Run Optuna Study

In [7]:
%%time
study = optuna.create_study(direction="maximize",storage="sqlite:///db.sqlite3",  # Specify the storage URL here.
        study_name="gradient_boosting_2")
study.optimize(objective, n_trials=25)

study.best_params, study.best_value


[I 2026-04-12 19:39:14,199] A new study created in RDB with name: gradient_boosting_2
c:\Users\MGarn\miniconda3\envs\analyst-lab\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
[I 2026-04-12 19:47:59,558] Trial 0 finished with value: 0.17671514849203315 and parameters: {'n_estimators': 157, 'max_depth': 2, 'learning_rate': 0.02627446483391999, 'subsample': 0.6677877849033309, 'max_features': None, 'min_samples_leaf': 12}. Best is trial 0 with value: 0.17671514849203315.
c:\Users\MGarn\miniconda3\envs\analyst-lab\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
[I 2026-04-12 20:33:03,906] Trial 1 finished with value: 0.17907526732354437 and parameters: {'n_estimators': 442, 'max_depth': 7, 'learning_rate': 0.0010446588623488895, 'subsample': 0.509559

CPU times: total: 1h 32min 21s
Wall time: 1h 35min 22s


KeyboardInterrupt: 

## 7. Train Final Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_model = clone(model)
best_model.set_params(**{
    f"classifier__{k}": v for k, v in study.best_params.items()
})

best_model.fit(X_train, y_train)
print(f'Best Model Score: {best_model.score(X_test, y_test):.3f}')
study.best_params


In [ ]:
best_model

In [ ]:
# Run in your terminal (not a Python cell):
# optuna-dashboard sqlite:///db.sqlite3
# Or uncomment the line below to launch from the notebook:
#!optuna-dashboard sqlite:///db.sqlite3
